# Aşama 4 — Spark SQL ile Sorgulama ve Analiz

Aşama 1'in temiz Parquet tablolarını geçici görünümler (temporary view) olarak kaydedip tüm analizi **saf SQL** ile yapıyoruz. Veri seti çok tablolu (işlem özellikleri, sınıflar, işlem→işlem kenarları, cüzdanlar, adres↔işlem kenarları), dolayısıyla JOIN'ler yapay değil, sorunun kendisinden doğuyor.

| Görünüm | Satır | Anahtar |
|---|---|---|
| `txs` | 203.769 | txId |
| `tx_edges` | 234.355 | src → dst (txId) |
| `wallets` | 920.691 | (address, time_step) |
| `wallet_classes` | 822.942 | address |
| `addr_tx` / `tx_addr` | 477.117 / 837.124 | address ↔ txId |

Her sorgunun ardından bulgu yorumlanıyor.

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))
from common.spark_utils import *  # get_spark, RAW, PROC, MODELS, CHECKPOINTS, TRAIN_MAX_STEP ...
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

In [2]:
spark = get_spark("04-spark-sql")
for name in ["txs", "tx_edges", "wallets", "wallet_classes", "addr_tx", "tx_addr"]:
    spark.read.parquet(str(PROC / name)).createOrReplaceTempView(name)
spark.sql("SHOW VIEWS").show()
sql = lambda q: spark.sql(q).toPandas()

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


+---------+--------------+-----------+
|namespace|      viewName|isTemporary|
+---------+--------------+-----------+
|         |       addr_tx|       true|
|         |       tx_addr|       true|
|         |      tx_edges|       true|
|         |           txs|       true|
|         |wallet_classes|       true|
|         |       wallets|       true|
+---------+--------------+-----------+



## S1 · Zaman içinde illicit aktivite (`GROUP BY`, `CASE`, `COUNT_IF`, `ORDER BY`)

In [3]:
s1 = sql("""
SELECT time_step,
       COUNT(*)                                   AS toplam,
       COUNT_IF(class_name = 'illicit')           AS illicit,
       COUNT_IF(class_name = 'licit')             AS licit,
       ROUND(COUNT_IF(class_name = 'illicit') / COUNT_IF(label IS NOT NULL), 4) AS illicit_orani,
       ROUND(SUM(out_BTC_total), 1)               AS btc_hacmi,
       CASE WHEN time_step <= 34 THEN 'eğitim' WHEN time_step < 43 THEN 'test-önce' ELSE 'test-sonra (≥43)' END AS donem
FROM txs
GROUP BY time_step
ORDER BY illicit_orani DESC
LIMIT 10
""")
s1

,time_step,toplam,illicit,licit,illicit_orani,btc_hacmi,donem
0,13,4528,291,518,0.3597,29341.5,eğitim
1,9,4996,248,530,0.3188,126743.4,eğitim
2,28,1653,85,199,0.2993,26137.2,eğitim
3,20,4291,260,640,0.2889,32987.6,eğitim
4,29,4275,329,845,0.2802,376780.6,eğitim
5,32,4525,342,981,0.2585,112564.0,eğitim
6,16,2975,128,402,0.2415,110795.7,eğitim
7,15,3639,147,471,0.2379,63845.6,eğitim
8,25,2314,118,476,0.1987,20730.9,eğitim
9,11,4296,131,565,0.1882,61230.5,eğitim


En yüksek illicit oranları 13 (%36), 9 (%32), 28, 20 ve 29. adımlarda; komşu adımlarda ise oran %2–10'a iniyor. Illicit aktivite zamana yayılmış sabit bir arka plan değil, **atak (burst) halinde** geliyor. En yüksek oranlı 10 adımın tamamı eğitim döneminde (≤34). Test dönemi, modelin eğitimde gördüğünden daha "sakin" bir dağılıma sahip.

## S2 · Illicit ve licit işlemlerin yapısal profili (toplulaştırma fonksiyonları)

In [4]:
sql("""
SELECT class_name,
       COUNT(*)                                       AS n,
       ROUND(AVG(num_input_addresses), 2)             AS ort_girdi_adres,
       ROUND(AVG(num_output_addresses), 2)            AS ort_cikti_adres,
       PERCENTILE_APPROX(num_output_addresses, 0.5)   AS medyan_cikti,
       ROUND(PERCENTILE_APPROX(out_BTC_total, 0.5), 4) AS medyan_btc,
       ROUND(PERCENTILE_APPROX(fee_rate_sat_b, 0.5), 1) AS medyan_ucret_sat_b,
       ROUND(AVG(size), 0)                            AS ort_boyut_byte,
       ROUND(AVG(CASE WHEN num_output_addresses = 2 THEN 1 ELSE 0 END), 3) AS iki_ciktili_orani,
       MAX(out_BTC_total)                             AS maks_btc
FROM txs
WHERE aug_missing = 0
GROUP BY class_name
ORDER BY class_name
""")

,class_name,n,ort_girdi_adres,ort_cikti_adres,medyan_cikti,medyan_btc,medyan_ucret_sat_b,ort_boyut_byte,iki_ciktili_orani,maks_btc
0,illicit,4545,2.77,1.35,1.0,0.1234,105.4,474.0,0.350,113.694893
1,licit,41500,5.20,6.17,2.0,0.4812,89.7,1072.0,0.755,6335.201666
2,unknown,156759,1.95,3.78,2.0,0.6917,96.0,475.0,0.827,11391.855937


Ortalamalar çarpık dağılımlarda yanıltıcı olduğu için medyan (`PERCENTILE_APPROX`) da raporlanıyor.

* Licit işlemlerin %75.5'i **iki çıktılı**. Bu, sıradan bir ödemenin imzası: "alıcıya ödeme + kendime para üstü". Illicit işlemlerde bu oran yalnızca %35; medyan çıktı sayısı **1**. Illicit işlemler çoğunlukla fonu bölmeden tek bir adrese **aktarıyor** (forwarding).
* Illicit işlemler daha küçük (medyan 0.12 BTC'ye karşı 0.48 BTC, ortalama boyut 474 B'ye karşı 1072 B) ve daha yüksek ücret oranı ödüyor (medyan 105'e karşı 90 sat/B). Hızlı onay için fazla ödeme yapılıyor olabilir.
* Maksimum değerler de anlamlı: En büyük illicit işlem 114 BTC, licit tarafta ise 6.335 BTC.

## S3 · Homofili: illicit işlemler kime para gönderiyor? (self-`JOIN` ×2 + pencere fonksiyonu)

Her etiketli kenar için kaynak ve hedefin sınıfı. `SUM(...) OVER (PARTITION BY ...)` ile satır yüzdesi.

In [5]:
sql("""
SELECT s.class_name AS kaynak, d.class_name AS hedef, COUNT(*) AS kenar,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY s.class_name), 4) AS kaynak_icindeki_pay
FROM tx_edges e
JOIN txs s ON e.src = s.txId
JOIN txs d ON e.dst = d.txId
WHERE s.label IS NOT NULL AND d.label IS NOT NULL
GROUP BY s.class_name, d.class_name
ORDER BY kaynak, hedef
""")

,kaynak,hedef,kenar,kaynak_icindeki_pay
0,illicit,illicit,998,0.5217
1,illicit,licit,915,0.4783
2,licit,illicit,781,0.0225
3,licit,licit,33930,0.9775


Etiketli verinin ~%10'u illicit. Buna karşılık illicit bir işlemden çıkan etiketli kenarların **%52.2'si yine illicit** bir işleme gidiyor (taban oranın ~5 katı). Licit işlemlerden illicit'e giden kenarların oranı ise yalnızca %2.25. Bu **güçlü homofili**, Aşama 2'de aggregate (komşuluk) özelliklerinin precision'ı neden belirgin biçimde artırdığını açıklıyor.

### Fiziksel plan: Spark bu JOIN'i nasıl çalıştırıyor?

In [6]:
plan = spark.sql("""
EXPLAIN FORMATTED
SELECT s.class_name, d.class_name, COUNT(*)
FROM tx_edges e JOIN txs s ON e.src = s.txId JOIN txs d ON e.dst = d.txId
WHERE s.label IS NOT NULL AND d.label IS NOT NULL
GROUP BY 1, 2
""").collect()[0][0]
blocks = plan.split("\n\n")
print(blocks[0])
print("\n" + next(b for b in blocks if b.startswith("(4) Scan parquet")))

== Physical Plan ==
AdaptiveSparkPlan (18)
+- HashAggregate (17)
   +- Exchange (16)
      +- HashAggregate (15)
         +- Project (14)
            +- BroadcastHashJoin Inner BuildRight (13)
               :- Project (8)
               :  +- BroadcastHashJoin Inner BuildLeft (7)
               :     :- BroadcastExchange (3)
               :     :  +- Filter (2)
               :     :     +- Scan parquet  (1)
               :     +- Project (6)
               :        +- Filter (5)
               :           +- Scan parquet  (4)
               +- BroadcastExchange (12)
                  +- Project (11)
                     +- Filter (10)
                        +- Scan parquet  (9)

(4) Scan parquet 
Output [3]: [txId#0L, class_name#197, label#198]
Batched: true
Location: InMemoryFileIndex [data/processed/txs]
PushedFilters: [IsNotNull(label), IsNotNull(txId)]
ReadSchema: struct<txId:bigint,class_name:string,label:double>


Plan üç şeyi gösteriyor: (1) **Column pruning:** `txs` Parquet'inden 199 sütunun yalnızca gerekli olanları okunuyor (`ReadSchema`). (2) **Predicate pushdown:** `label IS NOT NULL` filtresi dosya okuma aşamasına iniyor (`PushedFilters`), böylece Parquet satır grubu istatistikleriyle gereksiz veri hiç okunmuyor. (3) **Join stratejisi:** İki join de `BroadcastHashJoin`. Catalyst, küçük tarafı tüm executor'lara kopyalayıp büyük tarafı **hiç shuffle etmeden** birleştiriyor. Plandaki tek `Exchange` (shuffle), son `GROUP BY` için. AQE (`AdaptiveSparkPlan`) bu kararları çalışma zamanı istatistikleriyle yeniden verebiliyor.

## S4 · Illicit işlemlere dokunan cüzdanlar (`UNION ALL` + 3 tablolu `JOIN` + CTE)

Elliptic++'ta hem işlemlerin hem cüzdanların etiketi var. Illicit bir işleme gönderici veya alıcı olarak katılan cüzdanların kendi etiketleri ne diyor? Bu etiketler birbirinden bağımsız mı?

In [7]:
sql("""
WITH katilim AS (
    SELECT address, txId, 'gönderici' AS rol FROM addr_tx
    UNION ALL
    SELECT address, txId, 'alıcı'     AS rol FROM tx_addr
),
cuzdan_ozet AS (
    SELECT k.address,
           COUNT(DISTINCT k.txId)                          AS islem_sayisi,
           COUNT(DISTINCT IF(t.class_name = 'illicit', k.txId, NULL)) AS illicit_islem
    FROM katilim k JOIN txs t ON k.txId = t.txId
    GROUP BY k.address
)
SELECT CASE WHEN illicit_islem = 0 THEN '0 illicit işlem'
            WHEN illicit_islem = 1 THEN '1 illicit işlem'
            ELSE '≥2 illicit işlem' END AS grup,
       COUNT(*) AS cuzdan,
       ROUND(AVG(IF(wc.class_name = 'illicit', 1, 0)), 4) AS cuzdan_etiketi_illicit,
       ROUND(AVG(IF(wc.class_name = 'licit', 1, 0)), 4)   AS cuzdan_etiketi_licit,
       ROUND(AVG(IF(wc.class_name = 'unknown', 1, 0)), 4) AS cuzdan_etiketi_unknown
FROM cuzdan_ozet c JOIN wallet_classes wc ON c.address = wc.address
GROUP BY 1
ORDER BY 1
""")

,grup,cuzdan,cuzdan_etiketi_illicit,cuzdan_etiketi_licit,cuzdan_etiketi_unknown
0,0 illicit işlem,808676,0.0,0.3105,0.6895
1,1 illicit işlem,12932,1.0,0.0000,0.0000
2,≥2 illicit işlem,1334,1.0,0.0000,0.0000


**Bulgu: Etiketler bağımsız değil, türetilmiş.** En az bir illicit işleme dokunan 14.266 cüzdanın **%100'ü** illicit etiketli; illicit işleme hiç dokunmamış 808.676 cüzdanın **hiçbiri** illicit değil. Yani Elliptic++'ta cüzdan etiketi deterministik bir kuralla üretilmiş: "illicit bir işleme katıldıysan illicit'sin."

Bunun pratik bir sonucu var: Cüzdan etiketlerini (ya da onlardan türetilen bir özelliği) işlem sınıflandırmasında girdi olarak kullanmak **doğrudan etiket sızıntısı** olur. Aşama 2'de cüzdan bilgisini bu yüzden özellik olarak kullanmadık. SQL ile yapılan bu tek kontrol, veri setinin belgelerinde açıkça yazmayan bir inşa kuralını ortaya çıkardı.

## S5 · Her zaman adımının en büyük işlemleri (`RANK() OVER (PARTITION BY ... ORDER BY ...)`)

In [8]:
sql("""
SELECT time_step, sira, txId, class_name, ROUND(out_BTC_total, 2) AS btc, num_output_addresses
FROM (
    SELECT *, RANK() OVER (PARTITION BY time_step ORDER BY out_BTC_total DESC) AS sira
    FROM txs WHERE out_BTC_total IS NOT NULL
) WHERE sira <= 3 AND time_step IN (9, 13, 32, 43, 49)
ORDER BY time_step, sira
""")

,time_step,sira,txId,class_name,btc,num_output_addresses
0,9,1,175518212,unknown,2041.92,2.0
1,9,2,178921156,unknown,2020.92,2.0
2,9,3,179075436,unknown,572.74,19.0
3,13,1,339086046,unknown,452.81,2.0
4,13,2,339062612,licit,369.00,2.0
5,13,3,339062657,licit,366.00,2.0
6,32,1,355124275,unknown,11391.86,6.0
7,32,2,355004636,unknown,11311.83,6.0
8,32,3,355006792,unknown,11231.75,6.0
9,43,1,94071606,unknown,1295.04,6.0


In [9]:
sql("""
SELECT class_name,
       COUNT(*)                                         AS adim_basina_top10_kayit,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4)        AS pay
FROM (SELECT class_name, RANK() OVER (PARTITION BY time_step ORDER BY out_BTC_total DESC) r FROM txs)
WHERE r <= 10
GROUP BY class_name ORDER BY pay DESC
""")

,class_name,adim_basina_top10_kayit,pay
0,unknown,317,0.6469
1,licit,173,0.3531


Her adımın en büyük 10 işleminden oluşan 490 kaydın **hiçbiri illicit değil** (%65 unknown, %35 licit). Illicit akış büyük tek transferlerle değil, küçük transferlerle yürüyor. "Büyük miktar = şüpheli" sezgisi bu veride geçerli değil. 32. adımdaki ~11.300 BTC'lik ardışık transferler gibi en büyük hareketler, borsa veya saklama (custody) iç transferlerine benziyor.

## S6 · Cüzdan davranışı sınıfa göre (CTE + `JOIN` + toplulaştırma)

`wallets` tablosu (adres, zaman adımı) seviyesinde. Önce adres başına özetliyoruz, sonra sınıfla birleştiriyoruz.

In [10]:
sql("""
WITH adres AS (
    SELECT address,
           MAX(lifetime_in_blocks)          AS omur_blok,
           MAX(num_timesteps_appeared_in)   AS gorundugu_adim,
           MAX(total_txs)                   AS toplam_islem,
           MAX(btc_transacted_total)        AS btc_toplam,
           MAX(transacted_w_address_total)  AS etkilesilen_adres
    FROM wallets GROUP BY address
)
SELECT wc.class_name,
       COUNT(*)                                        AS cuzdan,
       PERCENTILE_APPROX(a.omur_blok, 0.5)             AS medyan_omur_blok,
       ROUND(AVG(a.gorundugu_adim), 2)                 AS ort_gorundugu_adim,
       ROUND(AVG(IF(a.toplam_islem = 1, 1, 0)), 3)     AS tek_kullanimlik_orani,
       ROUND(PERCENTILE_APPROX(a.btc_toplam, 0.5), 4)  AS medyan_btc,
       ROUND(AVG(a.etkilesilen_adres), 1)              AS ort_etkilesilen_adres
FROM adres a JOIN wallet_classes wc USING (address)
GROUP BY wc.class_name ORDER BY wc.class_name
""")

,class_name,cuzdan,medyan_omur_blok,ort_gorundugu_adim,tek_kullanimlik_orani,medyan_btc,ort_etkilesilen_adres
0,illicit,14266,1.0,1.03,0.332,0.0202,3.6
1,licit,251088,0.0,1.10,0.826,0.0108,10.8
2,unknown,557588,0.0,1.13,0.606,0.0352,5.2


Beklentinin tersine, illicit cüzdanlar **daha az** tek kullanımlık (%33'e karşı licit %83). Licit tarafta tek kullanımlık adres norm, çünkü modern cüzdan yazılımları her para üstü için yeni adres üretiyor. Illicit cüzdanlar ise aynı adresi birden fazla işlemde kullanıyor ve çok daha **dar bir çevreyle** etkileşiyor (ortalama 3.6'ya karşı 10.8 farklı adres). Para kapalı bir halka içinde dönüyor. Aşama 5'teki adres grafı homofilisi bu gözlemi doğruluyor.

## S7 · 43. adımdaki kırılma: önce ve sonra (`CTE` + `CASE` + toplulaştırma)

Aşama 2'deki model 43. adımdan sonra illicit işlemleri yakalayamıyordu. Illicit işlemlerin kendisi bu noktada değişti mi?

In [11]:
sql("""
WITH donem AS (
    SELECT *, CASE WHEN time_step < 43 THEN '1) adım 1-42' ELSE '2) adım 43-49' END AS d
    FROM txs WHERE label IS NOT NULL AND aug_missing = 0
)
SELECT d AS donem, class_name,
       COUNT(*) AS n,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d), 4)  AS donem_icindeki_pay,
       ROUND(PERCENTILE_APPROX(out_BTC_total, 0.5), 4)            AS medyan_btc,
       ROUND(AVG(num_input_addresses), 2)                         AS ort_girdi,
       ROUND(AVG(num_output_addresses), 2)                        AS ort_cikti,
       ROUND(PERCENTILE_APPROX(fee_rate_sat_b, 0.5), 1)           AS medyan_ucret,
       ROUND(AVG(Local_feature_1), 3) AS ort_LF1, ROUND(AVG(Aggregate_feature_1), 3) AS ort_AF1
FROM donem GROUP BY d, class_name ORDER BY d, class_name
""")

,donem,class_name,n,donem_icindeki_pay,medyan_btc,ort_girdi,ort_cikti,medyan_ucret,ort_LF1,ort_AF1
0,1) adım 1-42,illicit,4376,0.1109,0.1239,2.45,1.33,105.8,-0.166,-0.134
1,1) adım 1-42,licit,35072,0.8891,0.4891,5.08,6.28,66.0,-0.040,0.045
2,2) adım 43-49,illicit,169,0.0256,0.0935,11.07,2.00,98.3,-0.169,-0.104
3,2) adım 43-49,licit,6428,0.9744,0.4373,5.87,5.59,188.3,-0.035,0.061


43. adımdan sonra illicit işlemlerin hem **payı** (%11.1 → %2.6) hem de **profili** değişiyor. Ortalama girdi adresi sayısı **2.45'ten 11.07'ye** çıkıyor ve illicit işlemler artık licit işlemlerden (5.87) bile daha fazla girdi topluyor. Küçük forwarding işlemlerinden çok girdili **konsolidasyon** işlemlerine geçilmiş. Bu, literatürde 43. adım civarında büyük bir dark market'in kapanmasıyla açıklanan rejim değişimiyle tutarlı: Fonlar dağınık adreslerden toplanıp çıkarılıyor. Eğitimde "illicit = az girdili, tek çıktılı" diye öğrenen model bu yeni kalıbı licit bölgesine yerleştiriyor. Aşama 2'deki recall çöküşünün SQL ile gözlemlenmiş açıklaması bu.

## Özet

| # | Soru | Kullanılan SQL yapıları |
|---|---|---|
| S1 | Illicit aktivite zamanla nasıl değişiyor? | `GROUP BY`, `COUNT_IF`, `CASE`, `ORDER BY`, `LIMIT` |
| S2 | Illicit ve licit işlemler yapısal olarak nasıl farklı? | `AVG`, `MAX`, `PERCENTILE_APPROX`, `WHERE` |
| S3 | Illicit para nereye akıyor? | 2× self-`JOIN`, `SUM() OVER (PARTITION BY)`, `EXPLAIN` |
| S4 | İşlem etiketi ile cüzdan etiketi bağımsız mı? | `UNION ALL`, CTE, 3 tablolu `JOIN`, `COUNT(DISTINCT IF())` |
| S5 | En büyük işlemler kim? | `RANK() OVER (PARTITION BY ... ORDER BY ...)` |
| S6 | Illicit cüzdanlar nasıl davranıyor? | CTE, `JOIN USING`, toplulaştırma |
| S7 | 43. adımda ne değişti? | CTE, `CASE`, pencere payı |

In [12]:
spark.stop()